# Lab 1 - Supervised Fine-Tuning: contract review that cites its evidence

## Notebook 2 - Fine-tune with a serverless LoRA job

We now adapt **Nemotron 3 Nano 30B-A3B BF16** to the checklist using **LoRA** on SageMaker AI serverless customization: no training cluster to provision or container to build.

### Choosing a base model

The base model used across this lab is configured in [`config.py`](config.py). Want to
try a different model? Browse the available models on SageMaker JumpStart by running
the cell below. To switch, update `BASE_MODEL_ID` in `config.py` - all notebooks in
this lab pick up the change automatically.

> **Note:** Not every JumpStart model supports every customization technique. A
> "No recipes found" error means the model does not support SFT. Support for
> *serving* is a separate question - check that your chosen model can be served by
> the LMI/DJL container (notebook 4) before committing to it.


In [ ]:
import boto3

from config import BASE_MODEL_ID

sm = boto3.client("sagemaker")
models, kwargs = [], {"HubName": "SageMakerPublicHub", "HubContentType": "Model",
                      "MaxResults": 100}
while True:
    r = sm.list_hub_contents(**kwargs)
    for item in r["HubContentSummaries"]:
        if "@capability:customization" in item.get("HubContentSearchKeywords", []):
            models.append(item["HubContentName"])
    if "NextToken" in r:
        kwargs["NextToken"] = r["NextToken"]
    else:
        break

print(f"base model for this lab: {BASE_MODEL_ID}\n")
print(f"customizable models available ({len(models)}):")
print("\n".join(sorted(models)))

In [ ]:
%load_ext autoreload
%autoreload 2

from importlib.metadata import version
for package, expected in {"sagemaker-train": "1.21.0", "sagemaker-core": "2.21.0"}.items():
    actual = version(package)
    if actual != expected:
        raise RuntimeError(f"{package}=={actual}; install requirements.txt and restart the kernel (expected {expected}).")

#### Setup and dependencies

`MLFLOW_APP_ARN` in `config.py` is an optional override. If empty, setup first discovers the uniquely matching tagged stack app, ahead of a named workshop duplicate, before native SDK discovery/create. Ambiguity stops setup; creating/updating apps are waited for before training. No domain or tracking server is required and no account/domain defaults are changed. Optional `MLFLOW_ROLE_ARN` and `MLFLOW_ARTIFACT_STORE_URI` overrides select the named workshop reuse/create path; set both. Invalid configured ARNs and setup failures stop training. See the root README for permissions and creation defaults.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
from config import resolve_mlflow_app
mlflow_app_arn = resolve_mlflow_app(sm_client, session=sess)
print(f"MLflow App: {mlflow_app_arn}")
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from config import BASE_MODEL_ID, DATASET_PREFIX

base_model_id = BASE_MODEL_ID
training_dataset = DataSet.get(name=f"{DATASET_PREFIX}-train")
val_dataset = DataSet.get(name=f"{DATASET_PREFIX}-val")

output_path = (f"s3://{bucket_name}/{default_prefix}/{base_model_id}-contractnli"
               if default_prefix else f"s3://{bucket_name}/{base_model_id}-contractnli")
print(f"training output: {output_path}")

### Create the Model Package Group

The trained model is registered here, and notebooks 3 and 4 look it up by this name.

In [ ]:
import hashlib

from config import BASE_MODEL_ID

base_model_id = BASE_MODEL_ID

# SageMaker caps Model Package Group names at 63 characters. Hash-truncate when
# base_model_id + suffix exceeds it, so notebooks 2, 3 and 4 all derive the
# same name for any model id.
MAX_MPG_NAME_LENGTH = 63
suffix = "-contractnli-sft-mpg"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    # reserve room for the suffix, a hyphen separator, and the 6-char hash
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

print(f"Model Package Group: {model_package_group_name}")


In [ ]:
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

try:
    ModelPackageGroup.get(model_package_group_name=model_package_group_name)
    print(f"already exists: {model_package_group_name}")
except ClientError:
    ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="ContractNLI NDA checklist review - serverless SFT",
    )
    print(f"created: {model_package_group_name}")

### Configure the trainer

In [ ]:
from sagemaker.train.common import TrainingType
from sagemaker.train.sft_trainer import SFTTrainer

MAX_JOB_NAME_LENGTH, TIMESTAMP_LENGTH = 63, 15
base_job_name = "contractnli-sft"[: MAX_JOB_NAME_LENGTH - TIMESTAMP_LENGTH].rstrip("-")

trainer = SFTTrainer(
    model=base_model_id,
    training_type=TrainingType.LORA,
    model_package_group=model_package_group_name,
    mlflow_resource_arn=mlflow_app_arn,
    mlflow_experiment_name="contractnli-serverless-sft",
    training_dataset=training_dataset,
    validation_dataset=val_dataset,
    s3_output_path=output_path,
    sagemaker_session=sess,
    role=role,
    accept_eula=True,
    base_job_name=base_job_name,
)

print("default hyperparameters:")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")

### Hyperparameters

In [ ]:
trainer.hyperparameters.global_batch_size = 128
trainer.hyperparameters.max_epochs = 10
trainer.hyperparameters.learning_rate = 0.0001
trainer.hyperparameters.lr_warmup_steps_ratio = 0.1
trainer.hyperparameters.lora_rank = 32
trainer.hyperparameters.lora_alpha = 64

# Records longer than dataset_max_len (4096) are dropped, not truncated, so the
# effective dataset is smaller than what notebook 1 wrote. Counted with the
# NVIDIA-Nemotron-3-Nano-30B tokenizer; the training log confirms it as "Final trainset size: 315".
records, over_limit = 423, 108
kept = records - over_limit

gbs = int(trainer.hyperparameters.global_batch_size)
epochs = int(trainer.hyperparameters.max_epochs)
ratio = float(trainer.hyperparameters.lr_warmup_steps_ratio)

steps_per_epoch = -(-kept // gbs)     # the loader keeps the short final batch
scheduled = (kept // gbs) * epochs    # but the LR schedule is sized by floor division

print(f"{kept} of {records} records train ({over_limit} exceed dataset_max_len)")
print(f"{steps_per_epoch} steps/epoch x {epochs} epochs = {steps_per_epoch * epochs} optimizer steps")
print(f"cosine schedule sized for {scheduled} steps, "
      f"warmup = {ratio:.0%} = {round(ratio * scheduled)} steps\n")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")

### Launch

`wait=False` returns immediately. The job takes **20-25 minutes** for these 10 epochs.
A sizeable and fixed part of that is SageMaker provisioning the compute rather than
training, so the wall-clock does not scale with epochs the way you might expect -
halving the epochs does not halve the time.

In [ ]:
training_job = trainer.train(wait=False)
TRAINING_JOB_NAME = training_job.training_job_name
print(f"launched: {TRAINING_JOB_NAME}")

In [ ]:
region = sess.boto_region_name
console = f"https://{region}.console.aws.amazon.com"
print(f"Job: {console}/sagemaker/home?region={region}#/jobs/{TRAINING_JOB_NAME}")
print(f"Logs: {console}/cloudwatch/home?region={region}#logsV2:log-groups/log-group/"
      f"$252Faws$252Fsagemaker$252FTrainingJobs$3FlogStreamNameFilter$3D{TRAINING_JOB_NAME}")
url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_app_arn, ExpiresInSeconds=300)["AuthorizedUrl"]
print(f"MLflow (open within 5 minutes): {url}")
print("Select experiment: contractnli-serverless-sft")

Poll every 60 seconds until `Completed`, then continue to **notebook 3**. Failed or stopped jobs raise with their failure reason. Interrupting this cell does not stop the billable job. Keep `TRAINING_JOB_NAME` to reconnect after a kernel restart.

In Studio, open the configured MLflow App and select `contractnli-serverless-sft`, or use the temporary link above. Rerun the link cell if it expires. The experiment and metrics may appear only after provisioning and training startup; a missing curve immediately after launch is not a job failure. CloudWatch log streams also take time to appear.

In [ ]:
import time

started = time.monotonic()
while True:
    job = sm_client.describe_training_job(TrainingJobName=TRAINING_JOB_NAME)
    status, secondary = job["TrainingJobStatus"], job["SecondaryStatus"]
    print(f"[{(time.monotonic() - started) / 60:5.1f} min] {status} / {secondary}")
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(60)

if status != "Completed":
    reason = job.get('FailureReason') or 'No failure reason reported; inspect CloudWatch logs.'
    raise RuntimeError(f"{TRAINING_JOB_NAME}: {status} / {secondary}: {reason}")
print("Completed. Continue to notebook 3 for evaluation.")